## Imports and Helper Functions

In [1]:
# ============================================================
# Parameters cell  (mark this cell as "Parameters" in Fabric)
# ============================================================
mode            = "REPORTING"          # ETL | REPORTING | AUTO | GET | DISABLE
workspace_id    = ""              # Leave blank = use current workspace
etl_start_hour  = 22              # Only used when mode = "AUTO"
etl_end_hour    = 6
# ============================================================

In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import requests
import json

def get_token():
    return notebookutils.credentials.getToken("pbi")

def get_workspace_id():
    # Automatically uses the workspace where the notebook is running
    return notebookutils.runtime.context.get('currentWorkspaceId')

def get_sql_pools_url(workspace_id=None):
    if workspace_id is None:
        workspace_id = get_workspace_id()
    return f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/warehouses/sqlPoolsConfiguration?beta=true"

def get_headers():
    return {
        "Authorization": f"Bearer {get_token()}",
        "Content-Type": "application/json"
    }


## Get Current SQL Pool Config

In [3]:
def get_sql_pools_configuration():
    url = get_sql_pools_url()
    response = requests.get(url, headers=get_headers())
    if response.status_code == 200:
        config = response.json()
        print(json.dumps(config, indent=2))
        return config
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

# Run it
current_config = get_sql_pools_configuration()


{
  "customSQLPoolsEnabled": true,
  "customSQLPools": [
    {
      "name": "ETL",
      "isDefault": false,
      "optimizeForReads": false,
      "maxResourcePercentage": 70,
      "classifier": {
        "type": "Application Name",
        "value": [
          "ETL",
          "Load",
          "Pipeline",
          "DataFactory"
        ]
      }
    },
    {
      "name": "Reporting",
      "isDefault": false,
      "optimizeForReads": true,
      "maxResourcePercentage": 20,
      "classifier": {
        "type": "Application Name",
        "value": [
          "PowerBIPremium-DirectQuery",
          "Mashup Engine"
        ]
      }
    },
    {
      "name": "Default",
      "isDefault": true,
      "optimizeForReads": false,
      "maxResourcePercentage": 10,
      "classifier": {
        "type": "Application Name",
        "value": [
          "Default",
          "Unknown"
        ]
      }
    }
  ]
}


In [ ]:



import requests
import json
from datetime import datetime
# from notebookutils import notebook


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def get_token():
    return notebookutils.credentials.getToken("pbi")


def resolve_workspace_id():
    """Use the parameter if supplied, otherwise fall back to the current workspace."""
    if workspace_id and workspace_id.strip():
        return workspace_id.strip()
    return notebookutils.runtime.context.get('currentWorkspaceId')


def get_sql_pools_url(ws_id: str = None):
    if not ws_id:
        ws_id = resolve_workspace_id()
    return f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/warehouses/sqlPoolsConfiguration?beta=true"


def get_headers():
    return {
        "Authorization": f"Bearer {get_token()}",
        "Content-Type": "application/json"
    }


def log(msg: str):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")


# ------------------------------------------------------------
# Core API functions
# ------------------------------------------------------------
def get_sql_pools_configuration(ws_id: str = None):
    ws_id = ws_id or resolve_workspace_id()
    log(f"Getting SQL Pools configuration for workspace: {ws_id}")

    response = requests.get(get_sql_pools_url(ws_id), headers=get_headers())

    if response.status_code == 200:
        config = response.json()
        print(json.dumps(config, indent=2))
        return config
    else:
        log(f"ERROR {response.status_code}: {response.text}")
        raise Exception(f"Failed to get configuration: {response.status_code} - {response.text}")


def set_etl_mode(ws_id: str = None):
    ws_id = ws_id or resolve_workspace_id()
    log(f"Switching workspace {ws_id} to ETL mode (70% → ETL)...")

    body = {
        "customSQLPoolsEnabled": True,
        "customSQLPools": [
            {
                "name": "ETL",
                "isDefault": False,
                "maxResourcePercentage": 70,
                "optimizeForReads": False,
                "classifier": {
                    "type": "Application Name",
                    "value": ["ETL", "Load", "Pipeline", "DataFactory"]
                }
            },
            {
                "name": "Reporting",
                "isDefault": False,
                "maxResourcePercentage": 20,
                "optimizeForReads": True,
                "classifier": {
                    "type": "Application Name",
                    "value": ["PowerBIPremium-DirectQuery", "Mashup Engine"]
                }
            },
            {
                "name": "Default",
                "isDefault": True,
                "maxResourcePercentage": 10,
                "optimizeForReads": False,
                "classifier": {
                    "type": "Application Name",
                    "value": ["Default", "Unknown"]          # required by the API
                }
            }
        ]
    }

    response = requests.patch(get_sql_pools_url(ws_id), headers=get_headers(), json=body)

    if response.status_code == 200:
        log("Successfully switched to ETL mode")
        return True
    else:
        log(f"Failed: {response.status_code} - {response.text}")
        raise Exception(f"Failed to set ETL mode: {response.status_code}")


def set_reporting_mode(ws_id: str = None):
    ws_id = ws_id or resolve_workspace_id()
    log(f"Switching workspace {ws_id} to Reporting mode (80% → Reporting, optimised for reads)...")

    body = {
        "customSQLPoolsEnabled": True,
        "customSQLPools": [
            {
                "name": "ETL",
                "isDefault": False,
                "maxResourcePercentage": 10,
                "optimizeForReads": False,
                "classifier": {
                    "type": "Application Name",
                    "value": ["ETL", "Load", "Pipeline", "DataFactory"]
                }
            },
            {
                "name": "Reporting",
                "isDefault": False,
                "maxResourcePercentage": 80,
                "optimizeForReads": True,
                "classifier": {
                    "type": "Application Name",
                    "value": ["PowerBIPremium-DirectQuery", "Mashup Engine"]
                }
            },
            {
                "name": "Default",
                "isDefault": True,
                "maxResourcePercentage": 10,
                "optimizeForReads": False,
                "classifier": {
                    "type": "Application Name",
                    "value": ["Default", "Unknown"]          # required by the API
                }
            }
        ]
    }

    response = requests.patch(get_sql_pools_url(ws_id), headers=get_headers(), json=body)

    if response.status_code == 200:
        log("Successfully switched to Reporting mode")
        return True
    else:
        log(f"Failed: {response.status_code} - {response.text}")
        raise Exception(f"Failed to set Reporting mode: {response.status_code}")


def disable_custom_sql_pools(ws_id: str = None):
    ws_id = ws_id or resolve_workspace_id()
    log(f"Disabling Custom SQL Pools in workspace {ws_id}...")

    body = {
        "customSQLPoolsEnabled": False,
        "customSQLPools": []
    }

    response = requests.patch(get_sql_pools_url(ws_id), headers=get_headers(), json=body)

    if response.status_code == 200:
        log("Custom SQL Pools successfully disabled")
        return True
    else:
        log(f"Failed: {response.status_code} - {response.text}")
        raise Exception(f"Failed to disable: {response.status_code}")


# ------------------------------------------------------------
# Decision logic
# ------------------------------------------------------------
def decide_mode_from_time(start_hour: int = 22, end_hour: int = 6) -> str:
    current_hour = datetime.now().hour
    log(f"Current hour: {current_hour} | ETL window: {start_hour}:00 → {end_hour}:00")

    if start_hour > end_hour:          # crosses midnight
        if current_hour >= start_hour or current_hour < end_hour:
            return "ETL"
    else:
        if start_hour <= current_hour < end_hour:
            return "ETL"
    return "REPORTING"


def main():
    ws_id = resolve_workspace_id()
    log(f"Notebook started | mode='{mode}' | workspace_id='{ws_id}'")

    if mode.upper() == "AUTO":
        action = decide_mode_from_time(etl_start_hour, etl_end_hour)
        log(f"AUTO mode resolved to → {action}")
    else:
        action = mode.upper()

    if action == "ETL":
        set_etl_mode(ws_id)
        result = "ETL"
    elif action == "REPORTING":
        set_reporting_mode(ws_id)
        result = "REPORTING"
    elif action == "GET":
        get_sql_pools_configuration(ws_id)
        result = "GET"
    elif action == "DISABLE":
        disable_custom_sql_pools(ws_id)
        result = "DISABLE"
    else:
        raise ValueError(f"Unknown mode: '{mode}'. Valid values: ETL, REPORTING, AUTO, GET, DISABLE")

    log(f"Completed successfully. Final action: {result}")
    # notebook.exit(result)


# ------------------------------------------------------------
# Entry point
# ------------------------------------------------------------
if __name__ == "__main__":
    main()